# Modeling chemotherapy response
We want to model the likelihood of a positive response to chemotherapy (e.g., tumor reduction > 30%) in breast cancer patients. The predictors we are going to use include:

* **Age**: Patient age at the time of diagnosis.

* **Tumor Size**: Initial tumor diameter (cm).

* **ER Status**: Estrogen receptor status (positive/negative).

* **Genomic Risk Score**: A continuous score from genetic profiling.

Our objective is to model the probability (and its uncertainty) of a positive response to chemo, in order to then provide recommendations for next steps (e.g., proceed with chemo, consider alternatives etc.).

We have data from 150 patients across 4 hospitals. We will use a hierarchical structure for hospital-specific effects. You might think why do we need hospital-specific effects? Aren't all hospitals following the same protocols? Actually the answer is no. Some hospitals follow more aggressive or conservative chemotherapy protocols, which can lead to differences in dosage, combination regimens, or treatment intervals. Staff expertise and experience can also affect how chemotherapy is administered or adjusted.

In [1]:
# Imports
import pymc as pm
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import arviz as az
from scipy import stats
import os

train_data = pd.read_csv("tumor_data.csv")
n_hospitals = max(train_data["hospital"]) + 1

Our hierarchical model will have partial pooling for both intercepts and  slopes. In particular, our model is:

**Hyperpriors**:
\begin{align*}
\mu_\alpha &\sim \mathcal{N}(0, 5) \\
\sigma_\alpha &\sim \text{HalfNormal}(5) \\
\mu_{\beta_j} &\sim \mathcal{N}(0, 5), \quad j = 1, \dots, 4 \\
\sigma_{\beta_j} &\sim \text{HalfNormal}(5), \quad j = 1, \dots, 4
\end{align*}

**Hospital-level parameters:**
\begin{align*}
\alpha_h &\sim \mathcal{N}(\mu_\alpha, \sigma_\alpha), \quad h = 1, \dots, H \\
\beta_{h,j} &\sim \mathcal{N}(\mu_{\beta_j}, \sigma_{\beta_j}), \quad h = 1, \dots, H,\ j = 1, \dots, 4
\end{align*}

**Likelihood:**
\begin{align*}
\text{logit}(p_i) &= \alpha_{h_i} + \sum_{j=1}^4 \beta_{h_i,j} \cdot x_{ij} \\
y_i &\sim \text{Bernoulli}(p_i)
\end{align*}

In [2]:
# Standardize continuous predictors
train_data["age_std"] = (train_data["age"] - train_data["age"].mean()) / train_data["age"].std()
train_data["tumor_size_std"] = (train_data["tumor_size"] - train_data["tumor_size"].mean()) / train_data["tumor_size"].std()
train_data["genomic_score_std"] = (train_data["genomic_score"] - train_data["genomic_score"].mean()) / train_data["genomic_score"].std()

# Define and fit the hierarchical model
with pm.Model() as chemo_model:
    mu_alpha = pm.Normal("mu_alpha", mu=0, sigma=5)
    sigma_alpha = pm.HalfNormal("sigma_alpha", sigma=5)
    mu_beta = pm.Normal("mu_beta", mu=0, sigma=5, shape=4)  # 4 predictors/independent variables
    sigma_beta = pm.HalfNormal("sigma_beta", sigma=5, shape=4)

    alpha = pm.Normal("alpha", mu=mu_alpha, sigma=sigma_alpha, shape=n_hospitals)
    beta = pm.Normal("beta", mu=mu_beta, sigma=sigma_beta, shape=(n_hospitals, 4))

    logit_p = (alpha[train_data["hospital"]] +
               beta[train_data["hospital"], 0] * train_data["age_std"] +
               beta[train_data["hospital"], 1] * train_data["tumor_size_std"] +
               beta[train_data["hospital"], 2] * train_data["er_status"] +
               beta[train_data["hospital"], 3] * train_data["genomic_score_std"])

    response_obs = pm.Bernoulli("response_obs", logit_p=logit_p, observed=train_data["response"])

    trace = pm.sample(2000, tune=1000, target_accept=0.98, return_inferencedata=True)





Auto-assigning NUTS sampler...
Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [mu_alpha, sigma_alpha, mu_beta, sigma_beta, alpha, beta]


Sampling 4 chains for 1_000 tune and 2_000 draw iterations (4_000 + 8_000 draws total) took 946 seconds.
There were 24 divergences after tuning. Increase `target_accept` or reparameterize.
There were 52 divergences after tuning. Increase `target_accept` or reparameterize.
There were 44 divergences after tuning. Increase `target_accept` or reparameterize.
There were 36 divergences after tuning. Increase `target_accept` or reparameterize.


In [3]:
az.summary(trace)


,mean,sd,hdi_3%,hdi_97%,mcse_mean,mcse_sd,ess_bulk,ess_tail,r_hat
mu_alpha,-6.629,1.990,-10.478,-3.205,0.046,0.034,1927.0,2519.0,1.00
mu_beta[0],0.178,0.713,-1.265,1.398,0.014,0.012,3026.0,2467.0,1.00
mu_beta[1],-0.428,0.937,-2.336,1.139,0.018,0.015,3644.0,2900.0,1.00
mu_beta[2],4.102,2.003,0.621,7.965,0.046,0.034,1962.0,2380.0,1.00
mu_beta[3],-0.527,0.791,-2.081,0.858,0.014,0.012,3825.0,3055.0,1.00
alpha[0],-6.394,1.999,-10.158,-2.951,0.048,0.035,1904.0,2444.0,1.00
alpha[1],-7.318,2.141,-11.427,-3.780,0.052,0.037,1764.0,2495.0,1.00
alpha[2],-6.858,2.033,-10.829,-3.446,0.048,0.035,1937.0,2322.0,1.00
alpha[3],-6.967,2.038,-10.864,-3.603,0.049,0.035,1897.0,2386.0,1.00
"beta[0, 0]",0.225,0.528,-0.810,1.216,0.008,0.007,4405.0,3553.0,1.00


In [ ]:
az.plot_trace(trace)
plt.show()

As we can see from the traces, the chains are well mixed and the posterior distribution obtained from each of the chains are similar. Exception here are some of the hyperparameters and particularly the variance variables, where there seems to be some slow mixing.  Furthermore, the R-hat values are all 1 (or close to it), while also the effective sample sizes are all within acceptable range. One possibility here is to go back and rethink the priors for these variables, and/or increase the burn-in of the chains.

# Predictions

In what follows we will use this model to make predictions for 5 new patients. The predictions are going to give us a probability (and credible interval) for the patient responding to the chemotherapy treatment. However, this is not what we are directly interested on. We will then translate this to a follow-up action for the patient.

In [ ]:
new_data = pd.DataFrame({
    "hospital": [3, 3, 3, 2, 2, 2, 2, 0],
    "age": [60.6, 63.8, 69.9, 56.6, 61.9, 54.2, 20.5, 31.2],
    "tumor_size": [4.3, 2.18, 3.7, 6.6, 1.8, 1.5, 0.3, 0.1],
    "er_status": [0, 1, 1, 1, 0, 1, 1, 0],
    "genomic_score": [35.6 , 69.6, 6.1, 52.3, 32.4, 6.5, 4.2, 7.4]
})

# Standardize new data using training stats
new_data["age_std"] = (new_data["age"] - train_data["age"].mean()) / train_data["age"].std()
new_data["tumor_size_std"] = (new_data["tumor_size"] - train_data["tumor_size"].mean()) / train_data["tumor_size"].std()
new_data["genomic_score_std"] = (new_data["genomic_score"] - train_data["genomic_score"].mean()) / train_data["genomic_score"].std()

print("\nNew Patient Data:")
print(new_data[["hospital", "age", "tumor_size", "er_status", "genomic_score"]])

# Step 4: Predict response for new patients
with chemo_model:
    logit_p_new = (alpha[new_data["hospital"]] +
                   beta[new_data["hospital"], 0] * new_data["age_std"] +
                   beta[new_data["hospital"], 1] * new_data["tumor_size_std"] +
                   beta[new_data["hospital"], 2] * new_data["er_status"] +
                   beta[new_data["hospital"], 3] * new_data["genomic_score_std"])
    p_response_new = pm.Deterministic("p_response_new", pm.math.invlogit(logit_p_new))
    trace_new = pm.sample_posterior_predictive(trace, var_names=["p_response_new"])

In [ ]:

# Extract predictions
new_predictions = trace_new.posterior_predictive["p_response_new"].mean(dim=["chain", "draw"]).values
new_hdi = az.hdi(trace_new.posterior_predictive["p_response_new"], hdi_prob=0.95)

# Step 5: Clinical decision rules
def clinical_decision(prob):
    if prob > 0.6:
        return "Proceed with Chemotherapy"
    elif prob > 0.3:
        return "Consider Combination Therapy or Trials"
    else:
        return "Explore Alternative Treatments (e.g., surgery, immunotherapy)"

print("\nPredictions and Clinical Decisions for New Patients:")
for i, (pred, hdi_vals) in enumerate(zip(new_predictions, new_hdi['p_response_new'])):
    decision = clinical_decision(pred)
    print(f"Patient {i+1}: Response Prob = {pred:.3f}, 95% HDI = [{hdi_vals[0]:.3f}, {hdi_vals[1]:.3f}], Decision = {decision}")

# Step 6: Visualize predictions with decisions
plt.figure(figsize=(10, 6))
for i, (pred, hdi_vals) in enumerate(zip(new_predictions, new_hdi['p_response_new'])):
    plt.plot([i, i], hdi_vals, "b-", lw=2)
    plt.plot(i, pred, "bo")
    plt.text(i, pred + 0.05, clinical_decision(pred)[:10] + "...", ha="center", fontsize=8)
plt.xticks(range(len(new_data)), [f"Patient {i+1}" for i in range(len(new_data))])
plt.ylabel("Probability of Chemotherapy Response")
plt.title("Predicted Chemotherapy Response with Clinical Decisions")
plt.ylim(0, 1)
plt.show()

The clinical decision rule are summasrized below:

* $>$ 0.6: High likelihood of response—proceed with chemo.

* 0.3–0.6: Moderate likelihood—consider combination therapies or clinical trials.

* $<$ 0.3: Low likelihood—explore alternatives (e.g., surgery, immunotherapy).

We visualize lot shows each patient’s predicted probability with 95% HDI and decision labels. We observe that for patients  3, 6 and 7 while our recommendation is to consider a combination of therapies, the uncertainty associated with it is very high. However, as we have seen in class, when we are talking about real-world decisions, not every error is the same. So in this case, we want to err towards the more aggressive approach and explore other options instead (or on top) of chemotherapy.

**Note**: Thresholds are illustrative; real-world use would align with oncology guidelines.